# 07: Busca na web com o Gemini (Google Search)

As notícias do pipeline vêm de uma busca no Google feita pelo próprio Gemini. O helper é `pesquisar_na_web()`
(`fluxo_financeiro/tools/web_search.py`), usado pela tool `buscar_noticias_web` (Analyzer) e pelas buscas
do subgrafo de pesquisa do Investigator (`pesquisar_pergunta`).

**Grounding** ("ancoragem"): em vez de responder só com o que aprendeu no treino, o modelo faz buscas no Google
durante a resposta e escreve a partir das páginas encontradas. A resposta vem com os metadados das fontes usadas.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. O helper, direto

`pesquisar_na_web(prompt)` chama o Gemini com a ferramenta `google_search` ligada e devolve o texto seguido
de `Fontes: site1, site2...`.

In [2]:
from fluxo_financeiro.tools.web_search import MAX_FONTES, pesquisar_na_web

resposta = pesquisar_na_web("Quais foram as notícias mais recentes sobre a Vale (VALE3) na B3? Responda em 1 parágrafo.")
print(resposta)

As notícias mais recentes sobre a Vale (VALE3) giram em torno da divulgação do seu balanço do terceiro trimestre de 2026, que apontou uma queda de 53% no lucro líquido em comparação ao mesmo período do ano anterior, somando US$ 1,255 bilhão, pressionado pela redução nos preços do minério de ferro e pelo aumento dos custos operacionais. Além disso, a administração descartou o pagamento de dividendos extraordinários para o restante do ano devido a incertezas macroeconômicas e geopolíticas, sinalizando que deve priorizar programas de recompra de ações e manter as projeções de produção de minério de ferro entre 335 e 345 milhões de toneladas para o fechamento de 2026.

Fontes: opetroleo.com.br, visnoinvest.com.br, seudinheiro.com


## 2. Fontes só com o nome do site

O Google devolve, para cada fonte, um link de **redirecionamento** com cerca de 200 caracteres. Esses links
não ajudam a análise e iriam para o contexto de todos os agentes seguintes, gastando tokens em cada chamada.
Por isso o helper guarda só o nome do site e no máximo `MAX_FONTES` deles.

In [3]:
fontes = resposta.rsplit("Fontes:", 1)[-1].strip() if "Fontes:" in resposta else "(sem fontes)"
print(f"MAX_FONTES = {MAX_FONTES}")
print("Fontes devolvidas:", fontes)

MAX_FONTES = 5
Fontes devolvidas: opetroleo.com.br, visnoinvest.com.br, seudinheiro.com


## 3. Custo e faturamento

Nos modelos Gemini 3, a busca no Google só funciona com **faturamento ativo** no projeto da chave
(5.000 buscas grátis por mês, depois US$ 14 a cada 1.000). Sem faturamento, a API devolve erro 429 e as tools
transformam isso numa mensagem "Erro ao...", sem derrubar o pipeline. O crédito de boas-vindas do Google Cloud
não paga essa API; o crédito pré-pago do AI Studio paga.

## 4. As duas tools de notícia, lado a lado

A web é a busca padrão; a NewsAPI ficou de reserva porque quase nunca achava notícia em português sobre
ações brasileiras. Compare o que cada uma traz para a mesma empresa.

In [4]:
from fluxo_financeiro.tools.news import buscar_noticias, buscar_noticias_web

print("=== buscar_noticias_web ===")
print(buscar_noticias_web.invoke("Vale mineradora VALE3"))

2026-11-06 21:38:17,913 - fluxo_financeiro - INFO - [tool] buscar_noticias_web: query='Vale mineradora VALE3'


=== buscar_noticias_web ===


2026-11-06 21:38:21,176 - fluxo_financeiro - INFO - [tool] buscar_noticias_web: ok (1384 chars)


<conteudo_externo origem="google_search">
As ações da Vale (**VALE3**) negociadas na B3 ganharam forte fôlego recente, impulsionadas pelo retorno do apetite ao risco do investidor estrangeiro e pela valorização do dólar frente ao real, o que ajudou o Ibovespa a avançar. O movimento de recuperação ocorre mesmo após a mineradora reportar uma queda expressiva em seu lucro líquido, pressionada por margens apertadas e por fatores macroeconômicos como a cotação do petróleo e o cenário do minério de ferro.

Apesar do balanço fraco, a diretoria da companhia descartou dividendos extraordinários no momento, mas o mercado mantém o foco na sólida geração de caixa da empresa e na disciplina de alocação de capital. Analistas avaliam que a remuneração recorrente aos acionistas continua tendo fundamentos sólidos, amparada por um programa de recompra de ações em andamento.

Adicionalmente, a Vale apresentou suas projeções operacionais e financeiras de longo prazo, estimando uma produção de minério de f

In [5]:
print("=== buscar_noticias (NewsAPI) ===")
print(buscar_noticias.invoke("Vale mineradora"))

2026-11-06 21:38:21,185 - fluxo_financeiro - INFO - [tool] buscar_noticias: query='Vale mineradora'


=== buscar_noticias (NewsAPI) ===


2026-11-06 21:38:21,628 - fluxo_financeiro - INFO - [tool] buscar_noticias: 5 artigos encontrados


Notícias recentes sobre 'Vale mineradora':

<conteudo_externo origem="newsapi">
1. (2026-10-30) [Globo] Ibovespa cai 18% no pós-reeleição de Lula, maior perda semanal desde março de 2020
   A liquidação de ativos engatilhada pelo resultado eleitoral manteve o Ibovespa no negativo por mais um pregão nesta sexta-feira, o quinto consecutivo. Após oscilar entre os 167.008 e os 169.185 pontos, o índice terminou com recuo de 0,75%, aos 167.922 pontos,…
   [Contexto adicional]: A liquidação de ativos engatilhada pelo resultado eleitoral manteve o Ibovespa no negativo por mais um pregão nesta sexta-feira, o quinto consecutivo. Após oscilar entre os 167.008 e os 169.185 ponto…

2. (2026-10-30) [Globo] Volumes e custos do minério de ferro e da VBM estão dentro do esperado, afirma Vale
   O vice-presidente executivo de finanças e relações com investidores da Vale, Marcelo Bacci, afirmou nesta sexta-feira (30), em entrevista sobre os resultados do terceiro trimestre que os volumes e custos do miné

## O que observar

- O helper devolve texto puro; as tools é que acrescentam a tag `<conteudo_externo origem="google_search">` (ou `newsapi`), que diz ao agente que aquilo é dado de fora.
- A lista de fontes tem só nomes de sites, nunca links longos.
- A web traz um resumo já escrito; a NewsAPI traz artigos soltos (título, descrição, trecho), muitas vezes em inglês, marcados com `(EN)`.
- Cada chamada da busca na web é uma chamada ao Gemini com grounding: é o passo `busca_google` que aparece no LangSmith (notebook 11).